# Calling cuPIQP with CuPy, PyTorch and JAX arrays

cuPIQP is **Warp-native**: it stores all its data and results as `warp.array` objects. You don't need
to use Warp yourself, though. `setup()` and `update()` accept **any GPU array** that exposes
the [CUDA Array Interface](https://numba.readthedocs.io/en/stable/cuda/cuda_array_interface.html)
or [DLPack](https://dmlc.github.io/dlpack/latest/): CuPy, PyTorch, JAX, Numba, ... cuPIQP
reads them in place and copies their values into its own GPU storage. Nothing passes through
the host, so you may change or free your arrays afterwards.

The results come back as `warp.array` objects on the GPU, and each library can view them
without a copy:

| library | input to `setup` / `update` | result to the library |
|---|---|---|
| CuPy | `cupy.ndarray` | `cp.asarray(solver.result.x)` (zero-copy) |
| PyTorch | CUDA `torch.Tensor` | `torch.from_dlpack(solver.result.x)` (zero-copy) |
| JAX | `jax.Array` on a GPU | `jnp.from_dlpack(...)` of a contiguous copy, see below |

This notebook solves the same batch of QPs as [Getting started](getting_started.ipynb) once
with each library, then mixes all of them in a single call.

In [1]:
import os
# By default JAX reserves 75% of the GPU memory when it starts; let it allocate on demand
# instead, so that it shares the GPU with cuPIQP, CuPy and PyTorch.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import numpy as np
import cupy as cp
import torch
import jax
import jax.numpy as jnp
import warp as wp
from cupiqp import DenseSolver, SparseSolver, Status

wp.config.log_level = wp.LOG_WARNING  # hide Warp's start-up and kernel-loading messages
jax.config.update("jax_enable_x64", True)  # JAX uses float32 unless told otherwise

## Example QPs

We consider the following problem:

$$
\begin{aligned}
\min_{x_1,\,x_2}\quad & \tfrac12\bigl(6 x_1^2 + 4 x_2^2\bigr) - x_1 - 4 x_2 \\
\text{s.t.}\quad
  & x_1 - 2 x_2 = 1, \\
  & x_1 - x_2 \leq 0.2, \\
  & 2x_1 \le -1, \\
  & \theta \le x_1 \le 1,
\end{aligned}
$$

where $\theta$ is the lower bound on $x_1$ which varies among problems in the batch. We
create a batch of $B=4$ problems that differ only in $\theta$:

| problem | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| $\theta$ | $-2.0$ | $-1.5$ | $-1.0$ | $2.0$ |

Problem 3 is infeasible since the bounds of $x_1$ become $2 \leq x_1 \leq 1$, so the box
is empty.

The data is written once with NumPy, on the host. Every section below moves it to the GPU
with its own library. Arguments shared by the whole batch have the shape of one problem,
and the batched lower bound `x_l` of shape `(B, n)` sets the batch size (see
[Getting started](getting_started.ipynb) for the shared / batched convention).

In [2]:
B = 4  # batch size
theta = np.array([-2.0, -1.5, -1.0, 2.0])

qp = dict(
    # quadratic + linear cost
    P=np.array([[6.0, 0.0],
                [0.0, 4.0]]),
    c=np.array([-1.0, -4.0]),
    # equality constraint:  A x = b
    A=np.array([[1.0, -2.0]]),
    b=np.array([1.0]),
    # two-sided inequalities:  h_l <= G x <= h_u   (use -inf / +inf for one-sided)
    G=np.array([[1.0, -1.0],
                [2.0,  0.0]]),
    h_l=np.array([-np.inf, -np.inf]),
    h_u=np.array([0.2, -1.0]),
    # box bounds:  x_l <= x <= x_u; x_l is batched (B, n), one row per problem
    x_l=np.stack([theta, np.full(B, -np.inf)], axis=1),
    x_u=np.array([1.0, np.inf]),
)

def solve_dense(data):
    """Set up a DenseSolver on `data` (a dict of GPU arrays) and solve."""
    solver = DenseSolver()
    solver.setup(**data)
    solver.solve()
    return solver

def print_result(solver):
    info = solver.result.info.to_host()  # waits for the solve
    x = solver.result.x.numpy()
    for i in range(B):
        print(f"problem {i}:  theta = {theta[i]:4.1f}   {info.status[i].name:<26} x = {x[i]}")

## CuPy

`cp.asarray` copies the host data to the GPU. The `cupy.ndarray` objects go straight into
`setup`:

In [3]:
data_cp = {name: cp.asarray(value) for name, value in qp.items()}

solver_cp = solve_dense(data_cp)
print_result(solver_cp)

problem 0:  theta = -2.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 1:  theta = -1.5   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 2:  theta = -1.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 3:  theta =  2.0   CUPIQP_PRIMAL_INFEASIBLE   x = [0.71170958 0.18356021]


`solver.result.x` is a `warp.array`. It exposes the CUDA Array Interface, so `cp.asarray`
wraps it as a `cupy.ndarray` that shares the solver's memory, without a copy:

In [4]:
x_cp = cp.asarray(solver_cp.result.x)

print(f"{type(x_cp).__name__} of shape {x_cp.shape}")
print("same device memory?", x_cp.data.ptr == solver_cp.result.x.ptr)

ndarray of shape (4, 2)
same device memory? True


The view always shows the latest solution of this solver: the next `solve()` overwrites
it. Call `x_cp.copy()` to keep one.

`SparseSolver` takes `P`, `A`, `G` as CSR triples `(indptr, indices, values)`. A
`cupyx.scipy.sparse.csr_matrix` provides all three on the GPU:

In [5]:
from cupyx.scipy.sparse import csr_matrix

def csr_cp(M):
    M = csr_matrix(cp.asarray(M))
    return M.indptr, M.indices, M.data

data_cp_sparse = {**data_cp, "P": csr_cp(qp["P"]), "A": csr_cp(qp["A"]), "G": csr_cp(qp["G"])}

solver_cp_sparse = SparseSolver()
solver_cp_sparse.setup(**data_cp_sparse)
solver_cp_sparse.solve()
print_result(solver_cp_sparse)

problem 0:  theta = -2.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 1:  theta = -1.5   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 2:  theta = -1.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 3:  theta =  2.0   CUPIQP_PRIMAL_INFEASIBLE   x = [0.71170958 0.18356021]


## PyTorch

CUDA `torch.Tensor` objects are passed the same way:

In [6]:
device = torch.device("cuda")
data_torch = {name: torch.as_tensor(value, device=device) for name, value in qp.items()}

solver_torch = solve_dense(data_torch)
print_result(solver_torch)

problem 0:  theta = -2.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 1:  theta = -1.5   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 2:  theta = -1.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 3:  theta =  2.0   CUPIQP_PRIMAL_INFEASIBLE   x = [0.71170958 0.18356021]


`torch.from_dlpack` views a result as a `torch.Tensor`, again without a copy:

In [7]:
x_torch = torch.from_dlpack(solver_torch.result.x)

print(f"{type(x_torch).__name__} of shape {tuple(x_torch.shape)} on {x_torch.device}")
print("same device memory?", x_torch.data_ptr() == solver_torch.result.x.ptr)

Tensor of shape (4, 2) on cuda:0
same device memory? True


For `SparseSolver`, a 2-D [`torch.sparse_csr_tensor`](https://docs.pytorch.org/docs/stable/generated/torch.sparse_csr_tensor.html)
`T` gives the CSR triple `(T.crow_indices(), T.col_indices(), T.values())`:

In [8]:
import warnings
warnings.filterwarnings("ignore", message="Sparse CSR tensor support is in beta state")

def csr_torch(M):
    T = torch.as_tensor(M, device=device).to_sparse_csr()
    return T.crow_indices(), T.col_indices(), T.values()

data_torch_sparse = {**data_torch,
                     "P": csr_torch(qp["P"]), "A": csr_torch(qp["A"]), "G": csr_torch(qp["G"])}

solver_torch_sparse = SparseSolver()
solver_torch_sparse.setup(**data_torch_sparse)
solver_torch_sparse.solve()
print_result(solver_torch_sparse)

problem 0:  theta = -2.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 1:  theta = -1.5   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 2:  theta = -1.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 3:  theta =  2.0   CUPIQP_PRIMAL_INFEASIBLE   x = [0.71170958 0.18356021]


Tensors that require grad cannot be shared through the CUDA Array Interface or DLPack,
so pass `T.detach()` instead. To differentiate through the solution, use the PyTorch layers
in `cupiqp.torch` (`DenseSolverLayer`, `SparseSolverLayer`, `MultistageSolverLayer`)
instead of calling the solver directly.

## JAX

`jax.Array` objects on the GPU work as well. Two things differ from CuPy and PyTorch:

- **Streams.** CuPy and PyTorch run on the CUDA default stream by default, and the solver
  stream (`solver.stream`) is ordered with it automatically. JAX runs on its own streams, so
  wait for your JAX arrays with `block_until_ready()` before passing them in, and for the
  solve with `wp.synchronize_stream(solver.stream)` before JAX reads a result.
- **Strides.** The results are views into larger solver buffers, e.g. `x` is a set of rows
  of one buffer holding all primal variables. JAX only imports compact arrays through
  DLPack, so copy the result into a contiguous array first. As JAX arrays are immutable,
  that copy is what you want anyway: a view would change on the next `solve()`.

In [9]:
data_jax = {name: jnp.asarray(value) for name, value in qp.items()}
for value in data_jax.values():
    value.block_until_ready()

solver_jax = solve_dense(data_jax)
print_result(solver_jax)

problem 0:  theta = -2.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 1:  theta = -1.5   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 2:  theta = -1.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 3:  theta =  2.0   CUPIQP_PRIMAL_INFEASIBLE   x = [0.71170958 0.18356021]


In [10]:
print("x is contiguous?", solver_jax.result.x.is_contiguous)

with wp.ScopedStream(solver_jax.stream):    # copy on the solver stream, after the solve
    x_copy = wp.clone(solver_jax.result.x)  # contiguous (B, n) array
wp.synchronize_stream(solver_jax.stream)

x_jax = jnp.from_dlpack(x_copy)             # zero-copy view of the contiguous copy
print(f"{type(x_jax).__name__} of shape {x_jax.shape}")
print("same device memory as the copy?", x_jax.unsafe_buffer_pointer() == x_copy.ptr)

x is contiguous? False
ArrayImpl of shape (4, 2)
same device memory as the copy? True


## Mixing libraries

Each argument is converted on its own, so one call may mix arrays from different libraries
(and `warp.array` objects):

In [11]:
solver_mixed = solve_dense(dict(
    P=data_torch["P"],                                    # torch
    c=data_cp["c"],                                       # cupy
    A=data_jax["A"],                                      # jax
    b=wp.array(qp["b"], dtype=wp.float64, device="cuda"), # warp
    G=data_cp["G"],                                       # cupy
    h_u=data_torch["h_u"],                                # torch (h_l = -inf is omitted)
    x_l=data_jax["x_l"],                                  # jax
    x_u=data_torch["x_u"],                                # torch
))
print_result(solver_mixed)

problem 0:  theta = -2.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 1:  theta = -1.5   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 2:  theta = -1.0   CUPIQP_SOLVED              x = [-0.6 -0.8]
problem 3:  theta =  2.0   CUPIQP_PRIMAL_INFEASIBLE   x = [0.71170958 0.18356021]


## Verify the solution consistency

All solvers should report the same per-problem status and agree on the optimum of every
problem that solved.

In [12]:
solvers = {
    "cupy dense": solver_cp, "cupy sparse": solver_cp_sparse,
    "torch dense": solver_torch, "torch sparse": solver_torch_sparse,
    "jax dense": solver_jax, "mixed dense": solver_mixed,
}
ref_status = solver_cp.result.info.status.numpy()
ref_x = solver_cp.result.x.numpy()
solved = ref_status == Status.CUPIQP_SOLVED

for name, solver in solvers.items():
    assert (solver.result.info.status.numpy() == ref_status).all(), name
    assert np.allclose(solver.result.x.numpy()[solved], ref_x[solved], atol=1e-6), name

print(f"all {len(solvers)} solvers agree on every problem.")

all 6 solvers agree on every problem.


## What is not accepted

cuPIQP never moves data between host and device or changes dtypes behind your back, so
two kinds of input raise a `TypeError`:

- **Host arrays** (NumPy, CPU tensors, ...): move them to the GPU first.
- **A different dtype** than the solver's (`float64` by default, `DenseSolver(dtype=wp.float32)`
  for single precision): convert the array first.

In [13]:
try:
    DenseSolver().setup(P=qp["P"], c=qp["c"])  # NumPy arrays live on the host
except TypeError as e:
    print("TypeError:", e)

try:
    DenseSolver().setup(P=data_torch["P"].float(), c=data_torch["c"])  # float32 P
except TypeError as e:
    print("TypeError:", e)

TypeError: DenseSolver requires P to be a GPU dense array (a warp.array, cupy.ndarray, dense CUDA torch.Tensor, JAX CUDA array, etc.); got ndarray.
TypeError: P has dtype float32; this solver uses float64. cuPIQP does not cast dtypes implicitly; convert P to float64 first.
